# Yeast Protein Sequence Retrieval

>Retrieves reviewed amino acid sequences from UniProt for all yeast proteins in the PPI dataset 
> and exports a sequence-annotated CSV.

In [1]:
import pandas as pd
import requests
from tqdm import tqdm

## Configuration and Paths

In [ ]:
INPUT_FILE  = "/path/to/0_All_protein_yeast.csv"
OUTPUT_FILE = "/path/to/1_All_protein_uniprot_yeast.csv"

PROT_COL = "gene_name"
TAXON_ID = "4932"  # Saccharomyces cerevisiae

## UniProt Proteome Retrieval

In [10]:
def get_yeast_database():
    """
    Downloads the reviewed Saccharomyces cerevisiae proteome from UniProt.
    Returns a dictionary mapping Gene symbols/aliases to amino acid sequences.
    """
    
    url = "https://rest.uniprot.org/uniprotkb/stream"
    params = {
        'format': 'tsv',
        'fields': 'gene_names,sequence',
        'query': f'taxonomy_id:{TAXON_ID} AND reviewed:true'
    }
    
    response = requests.get(url, params=params)
    response.raise_for_status()
    
    local_map = {}
    lines = response.text.strip().split('\n')
    
    for line in lines[1:]:
        parts = line.split('\t')
        if len(parts) >= 2:
            aliases = parts[0].upper().split() 
            sequence = parts[1]
            for alias in aliases:
                local_map[alias] = sequence
                
    return local_map

## Load & Pre-process 

In [11]:
yeast_db = get_yeast_database()
df = pd.read_csv(INPUT_FILE)

## Sequence Mapping

In [12]:
unique_genes = pd.concat([df[PROT_COL]]).unique()
gene_to_seq_map = {}

for gene in tqdm(unique_genes, desc="Mapping sequences"):
    if pd.isna(gene):
        gene_to_seq_map[gene] = "NOT_FOUND"
    else:
        name = str(gene).strip().upper()
        gene_to_seq_map[gene] = yeast_db.get(name, "NOT_FOUND")

df['seq_prot'] = df[PROT_COL].map(gene_to_seq_map)

Mapping sequences: 100%|██████████| 6039/6039 [00:00<00:00, 988850.36it/s]


## Quality Check & Export

In [ ]:
n_missing = (df["seq_prot"] == "NOT_FOUND").sum()
print(f"Sequences not found: {n_missing} / {len(df)}")

n_missing

df_clean = df[df["seq_prot"] != "NOT_FOUND"].copy()
print(f"Proteins retained:   {len(df_clean)}")

df_clean.to_csv(OUTPUT_FILE, index=False)
print(f"Saved → {OUTPUT_FILE} :)")

Sequences not found: 1 / 6039
Proteins retained:   6038
Saved → /Users/matteo/Desktop/UTTOPIA/B_Dataset_Yeast/1_All_protein_uniprot_yeast.csv :)


In [17]:
n_missing = (df["seq_prot"] == "NOT_FOUND").sum()
print(f"Sequences not found: {n_missing} / {len(df)}")


n = df[df["seq_prot"] == "NOT_FOUND"]
print(n)


Sequences not found: 1 / 6039
     gene_name   seq_prot
3555   YKR059W  NOT_FOUND
